# Sports Car Dataset Project - Cleaning Data
 I have taken a dataset containing infromation about popular sports cars (e.g. Porsche 911, Audi R8, etc) from Kaggle (`sports_car.csv`). This is a Large Language Model (LLM) generated csv file. I will use this to demonstrate data cleaning, exploration and visualisation. 
 
In this Juypter Notebook I will assess and clean the data using Pandas (formatting, missing values, data type correction etc). In another Notebook, I will go onto explore and visualise the data using pandas and matplotlib. I will also perform some SQL queries on sql lite, and create a dashboard of the data in Tableau. 
## Initial Inspection of Dataset
I firstly imported the relevant pandas and numpy libraries, and then loaded the dataset into a DataFrame (`df`). I previewed the DataFrame using `.head()` which showed the general layout of the dataset, and then `.info()` to look at data type and missing values. 
- **Column Names**: the column names, while informative (e.g. `0-60 MPH Time (seconds)`) , will be difficult to use in Pandas (risk of misspellings etc). I renamed the columns to make them cleaner, shorter and easier to type (e.g. `Car Make` was changed to `make`).
- **Incorrect Data Type**: the datatypes for some columns were incorrect. Such as `zero_to_sixty` and `price_usd` should be floats, and `horsepower` and `torque_lb`should be integers. These will need to be changed to enable calculations and visualisations to be performed. 
- **Missing Data**: there are some missing values in the `engine_size_l` and `torque_lb` category which need further investigation (possible imputation or deletion depending on reasoning).
- **Price Column**: the price column is in USD ($). As we are in the UK, I would like to convert this into GBP (£). To do this I will take the exchange rate and multiply this with the `price_usd` to create a `price_gbp` column. The current exchange rate at the time of this project is 0.76 GB Pounds to 1 US dollar. 

In [1]:
import pandas as pd
import numpy as np


# create df using the csv
df = pd.read_csv("sports_car.csv")


# preview 
print(df.head())
print(df.info())


# rename columns (easier to use)

df = df.rename({"Car Make": "make", 
                "Car Model": "model", 
                "Year": "year", 
                "Engine Size (L)": "engine_size_l", 
                "Horsepower": "horsepower", 
                "Torque (lb-ft)":"torque_lb",
                "0-60 MPH Time (seconds)": "zero_to_sixty",
                "Price (in USD)": "price_usd"
               }, axis=1)

# check column names 
print(df.columns)

      Car Make Car Model  Year Engine Size (L) Horsepower Torque (lb-ft)  \
0      Porsche       911  2022               3        379            331   
1  Lamborghini   Huracan  2021             5.2        630            443   
2      Ferrari   488 GTB  2022             3.9        661            561   
3         Audi        R8  2022             5.2        562            406   
4      McLaren      720S  2021               4        710            568   

  0-60 MPH Time (seconds) Price (in USD)  
0                       4        101,200  
1                     2.8        274,390  
2                       3        333,750  
3                     3.2        142,700  
4                     2.7        298,000  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Car Make                 1007 non-null   object
 1   Car Model      

## Dropping duplicates 
The first step is to remove any obvious duplicates in this data. I will be using the `make` and `model` as well as the `year` and other specs (`horsepower`, `torque`, `engine_size_l` and `price_usd`). This is because there can be multiple models of the same car released, even in the same year (e.g. one may have a 2 liter engine, another may have a 4 liter engine - both having different HP and torque). I found that there were quite a few duplicate values (366 duplicates). After removing them, I found that there were some minor changes in number of null values (e.g. only a few missing torque and engine_size). 

In [2]:
# identify duplicates 
duplicates_first = df.duplicated(subset=['make', 'model', 'year', 'engine_size_l', 'price_usd', 'horsepower', 'torque_lb'])

print(duplicates_first.value_counts())

df[duplicates_first]

# drop duplicates

df.drop_duplicates(
    subset=['make', 'model', 'year', 'engine_size_l', 'price_usd', 'horsepower', 'torque_lb'],
    keep='first',   # or 'last'
    inplace=True)

# view table after duplicates removed
print(df.info())

False    641
True     366
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 641 entries, 0 to 1006
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   make           641 non-null    object
 1   model          641 non-null    object
 2   year           641 non-null    int64 
 3   engine_size_l  632 non-null    object
 4   horsepower     641 non-null    object
 5   torque_lb      638 non-null    object
 6   zero_to_sixty  641 non-null    object
 7   price_usd      641 non-null    object
dtypes: int64(1), object(7)
memory usage: 45.1+ KB
None


## Assessing Each Column Individually 
After doing an initial inspection of the dataset, and identifying possible issues (missing data, incorrect data types etc), I will go through each column individually. Assessing (and rectifying if necessary) the formatting, data type and any missing data. 
### Assessing `make` Column
The first column I will look at is the `make` column. This is the car manufacterers name, using `nunique()` I can see that there are 38 unique values in the `make` column, and some examples are: Porsche, Lamboghini, Ferrari, Subaru and TVR. There are no missing values in this column, and we can see that the data type is correct (`object`). As there is no missing data, the data type is correct and the car make is correctly formatted, I can move onto the next column. 

In [3]:
make_num_unique = df.make.nunique()

print(f"There are {make_num_unique} unique Car Makes in this dataset")

print("\nExamples include:")
print(df.make.unique())

print("\nThe number of missing values in make column are:")
print(df.make.isna().sum())

print("\nThe data type is:")
print(df.make.dtype)

There are 38 unique Car Makes in this dataset

Examples include:
['Porsche' 'Lamborghini' 'Ferrari' 'Audi' 'McLaren' 'BMW' 'Mercedes-Benz'
 'Chevrolet' 'Ford' 'Nissan' 'Aston Martin' 'Bugatti' 'Dodge' 'Jaguar'
 'Koenigsegg' 'Lexus' 'Lotus' 'Maserati' 'Alfa Romeo' 'Ariel' 'Bentley'
 'Mercedes-AMG' 'Pagani' 'Polestar' 'Rimac' 'Acura' 'Mazda' 'Rolls-Royce'
 'Tesla' 'Toyota' 'W Motors' 'Shelby' 'TVR' 'Subaru' 'Pininfarina' 'Kia'
 'Alpine' 'Ultima']

The number of missing values in make column are:
0

The data type is:
object


### Assessing `model` Column
The next step is to look at the model column. This includes information about the model of each car. Using `unique()` I can see that there are 176 unqiue car models in the data set, and some examples are: 911, Huracan, R8, AMG GT and Corvette. There is no missing data in this column and the data type is correct (`object` - includes both letter and numbers in Model Names). The formatting appears to be correct as well (is unique to each model e.g. Huracan should be in title case, whereas AMG GT should be all uppercase - they all appear to be correct). As the formatting and data types is correct, and there is no missing data, I can move onto the next column. 

In [4]:
# unique values 

model_num_unique = df.model.nunique()

print(f"There are {model_num_unique} unqiue models in the dataset.")

print("\nSome examples include:")
print(df.model.unique()[:10])

# missing values 
print("\nThe number of missing values in Model column are:")
print(df.model.isna().sum())

# data type 
print("\nThe data type is:")
print(df.model.dtype)

There are 176 unqiue models in the dataset.

Some examples include:
['911' 'Huracan' '488 GTB' 'R8' '720S' 'M8' 'AMG GT' 'Corvette'
 'Mustang Shelby GT500' 'GT-R Nismo']

The number of missing values in Model column are:
0

The data type is:
object


### Assessing the `year` column 
The next step is to look at the `year` column, which contains the year of production for each car in the dataset. We can see that there are only 9 different years, ranging between 1965 and 2023. Using `unique()` we can see that most of the dates are more recent (2014 and later). This is further supported by the histogram showing the distribution of car production dates, with the vast majority occuring after 2020. There is one major outlier in the data, produced in 1965 which is the Shelby Cobra (the range of the histogram has been altered - outlier is not visable in it). \
There is no missing values in this column, but there is a posible issue with the data type. The year column, while it does appear at first to be a numeric column, it is actually a categorical. It contains discrete year labels, and doesn't behave in the same way as continuous (not in this data set anyway). Therefore, I will correct the data type and convert it to an ordered cateogirical column. After sorting the data type, there is nothing more that needs to be done with the `year` column, so I can move onto the next one. 

In [5]:
from matplotlib import pyplot as plt

# unique values 
year_num_unique = df.year.nunique()

# range 
min_year = df.year.min()

max_year = df.year.max()

print(f"There are {year_num_unique} unique years, ranging from {min_year} to {max_year}.")

print("\nExamples include:")
print(df.year.unique())


# outlier - 1965
print("these are the outliers:")
print(df[df["year"] == 1965])

# missing values 

print("\nThe number of missing values in Year column are:")
print(df.year.isna().sum())

print("\nThe data type of the Year Column is:")
print(df.year.dtype)

There are 9 unique years, ranging from 1965 to 2023.

Examples include:
[2022 2021 2015 2020 2019 2017 1965 2014 2023]
these are the outliers:
       make  model  year engine_size_l horsepower torque_lb zero_to_sixty  \
170  Shelby  Cobra  1965             7        435       440           4.2   

     price_usd  
170  1,000,000  

The number of missing values in Year column are:
0

The data type of the Year Column is:
int64


In [6]:
# convert to categorical (ordered)
df['year_clean'] = df['year'].astype('category')
df['year_clean'] = df['year_clean'].cat.as_ordered()



### Assessing `engine_size_l` column
The next step is to assess the `engine_size_l` column, which contains both numeric and number values of the engine size/ type. Using `nunique()` I can see that there are 45 different engine sizes in this data set, many of which are numeric (e.g. Porsche which is 3 liter) but some which are strings (e.g. Electric, Hybrid). \
There are 9 missing values in this data, and I also noticed a few placeholders like a dash and a zero being used to represent missing data, which means the total is likely greater than initially thought. \
The data type for the `engine_size_l` column is object, which makes sense given that there is a mix of numeric and string values. 

In [7]:
# unique 
engine_num_unique = df.engine_size_l.nunique()

print(f"There are {engine_num_unique} unique engine sizes in this data.")

print("\nSome examples are:")
print(df.engine_size_l.unique())

# missing data 


print("\nThe number of missing values in engine size column are:")
print(df.engine_size_l.isna().sum())


# data type

print("\nThe data type of the engine size is:")
print(df.engine_size_l.dtype)



There are 45 unique engine sizes in this data.

Some examples are:
['3' '5.2' '3.9' '4' '4.4' '6.2' '3.8' '8' '5' '3.5' '4.7' '2' '2.9' '6'
 'Electric' '6.5' '3.7' 'Electric Motor' '2.5' '1.5 + Electric' '6.8'
 '8.4' nan '6.6' '7' '1.7' '3.3' '-' '6.7' '1.8' 'Electric (tri-motor)'
 '5.5' 'Electric (93 kWh)' 'Electric (100 kWh)' 'Hybrid (4.0)' '4.6' '3.6'
 '1.5' 'Hybrid' '5.7' '2.0 (Electric)' '4.0 (Hybrid)' '0' '6.4' '6.3'
 '2.3']

The number of missing values in engine size column are:
9

The data type of the engine size is:
object


There were quite a few issues with the engine size column, I will address each of them:
1. **Placeholder values** - some missing values were filled in with placeholders (e.g. dash or zero). I will remove these so that we can get an accurate representation of missingness in the data.
2. **New Numeric Engine Size Column** - will create a numeric column for engine size, where all non-numeric values are converted to NaN. This will mean that only combustion engines (petrol or diesel) are included in the data, and all others (hybrid or electric) are represented by NaN.
3. **New Engine Type Column** - will then create a new numeric column for engine type, which will be split into three categories: combustion (any completely numerical engine size - indicates either diesel or petrol combustion engine), hybrid (includes hybrid in description) or electric (includes electric in description). Anything else will be converted to NaN.
4. **Assess missingness** - see if there is a pattern in missingness e.g. common across certain model or make, certain year.

#### Placeholder Removal 
Have used `.replace()` to remove any placeholders (dash or zero) and replace them with NaN. A quick check of the number of missing values shows that after placeholder removal the total has increased to 11. 

In [8]:
# placeholder removal 

placeholder = ["0", "-"]

df["engine_size_clean"] = df["engine_size_l"].replace(placeholder, np.NaN)

print("The new missing values in the engine size column is:")
print(df["engine_size_clean"].isna().sum())



The new missing values in the engine size column is:
11


#### Creating Numeric Engine Size Column 
Next I created a new numeric engine size column using `pd.to_numeric`, which took any values which were numeric and put them into a new column ('numeric_engine_size`) and any non-numeric values were converted to NaN. A check of null values reveals that there are 52 missing values in this new column (11 of which are due to being missing from engine size column - the rest are just non-combustion engines). I can also see that the data type has successfully been converted into a float. \
*It is important to note that this column will only include values for those which are combustion engines - therefore findings are only applicable to combustion cars*. 

In [9]:
# create numeric engine size column 

df["numeric_engine_size"] = pd.to_numeric(df["engine_size_clean"], errors="coerce")

print("Missing values in this new numeric column are:")
print(df.numeric_engine_size.isna().sum())

print("\nThe data type of this column is:")
print(df.numeric_engine_size.dtype)

Missing values in this new numeric column are:
52

The data type of this column is:
float64


### Create Engine Type Column 
Next I wanted to create an engine_type column which would use the `engine_size_clean` column to split data into three columns:
- Combustion - if the engine size was a numerical variable (e.g. 2.0, 3.0) - this suggests it is a petrol or diesel engine - due to it's size being measured in cylinder capacity.
- Hybrid - if the engine size contained the word 'hybrid' then it would be split into that cateogory.
- Electric - if the engine size contained the word 'electric' then it would be split into that cateogory.
- Unknown - any values which are missing in engine size column, or aren't assigned to the above categories will put in the unknown cateogory.

After creating the column, I found that there were no missing values (meaning all missing values have successfully been converted to unknown) and the data type was correct (object). Finally, I used `value_counts()` to assess the category size, finding that the majority are combustion (91%), then electric (6%), Unknown (2%) and hybrid (less than 0.5%). The unknown category has a count of 11, which is the same as our missing values from the engine size column (shows all other values have been correctly assigned to category). All other counts add up with the engine size column as well. 

In [10]:
# create engine type column 

def engine_type_classify(row):
    if pd.isna(row):
        return "Unknown"
    row_string = str(row).lower().strip()
    if "electric" in row_string:
        return "Electric"
    elif "hybrid" in row_string:
        return "Hybrid"
    elif row_string.replace('.', '', 1).isdigit():
        return "Combustion"
    else:
        return "Unknown"

# apply the classification 

df["engine_type"] = df['engine_size_clean'].apply(engine_type_classify)

# change to category 
df['engine_type'] = df['engine_type'].astype('category')

# missing values 
print("The number of missing values is:")
print(df.engine_type.isna().sum())

# data type check 
print("\nThe data type is:")
print(df.engine_type.dtype)


# create series of value counts and proportion 
counts = df['engine_type'].value_counts(dropna=False)
props = df['engine_type'].value_counts(normalize=True, dropna=False)

result = pd.DataFrame({
    'count': counts,
    'proportion': props
})

print("\nValue counts for each category:")
print(result)


The number of missing values is:
0

The data type is:
category

Value counts for each category:
             count  proportion
engine_type                   
Combustion     589    0.918877
Electric        37    0.057722
Unknown         11    0.017161
Hybrid           4    0.006240


#### Assess Missing Values 
I have inspected the missing values, finding the majority of them are produced by Tesla or Rimac, both companies produce completely electric cars. In addition to this, a quick search comfirmed that the other three: Porsche Taycan (and Taycan Turbo S) and the Lotus Evija are all electric cars. Therefore, we can change these in the `engine_type` category to "Electric". - to do this I replaced all "Unknown" category with "Electric". This is only because I know all of the Unknowns are the ones with missing values (see below) - an alternative would be to use a mask/ only replace specific entries, this is not needed in this case. \
A quick check on the value counts and proportions shows that "Electric" now accounts for 7% of the cars in our dataset. 

In [11]:
# inspecting missing engine size values 
missing_engine_value = df[df["engine_size_clean"].isna()]

print(missing_engine_value)

# replacing all Unknown, with "Electric"

df["engine_type"] = df["engine_type"].replace("Unknown", "Electric")


# create series of value counts and proportion - new
counts = df['engine_type'].value_counts(dropna=False)
props = df['engine_type'].value_counts(normalize=True, dropna=False)

result = pd.DataFrame({
    'count': counts,
    'proportion': props
})

print("\nValue counts for each category:")
print(result)

        make           model  year engine_size_l horsepower torque_lb  \
168    Rimac           C_Two  2022           NaN       1914      1696   
171    Tesla   Model S Plaid  2021           NaN       1020      1050   
222  Porsche  Taycan Turbo S  2021           NaN        750       774   
247    Tesla   Model S Plaid  2022           NaN       1020      1050   
335    Tesla        Roadster  2022             -      1000+         -   
387    Rimac           C_Two  2022           NaN       1888      1696   
389    Tesla        Roadster  2022           NaN     10000+         0   
697    Lotus           Evija  2022           NaN       1972      1254   
752  Porsche          Taycan  2022           NaN        469       479   
885    Tesla        Roadster  2022             0     10,000     7,376   
916    Tesla        Roadster  2022           NaN    10,000+       NaN   

    zero_to_sixty  price_usd year_clean engine_size_clean  \
168           1.9  2,400,000       2022               NaN   
1

In [12]:
# side by side of new columns vs old 

print(df[["engine_size_l", "numeric_engine_size", "engine_type"]].sample(5))


    engine_size_l  numeric_engine_size engine_type
933           3.5                  3.5  Combustion
88              5                  5.0  Combustion
596             3                  3.0  Combustion
28            6.5                  6.5  Combustion
187           3.9                  3.9  Combustion


### Assess `horsepower` Column 
The next step is to assess the horsepower, which is the power output of the car. Looking at the unique values of the horsepower, I noticed that there are a few which are rough estimates, rather than exact values e.g. 1000+ and 10,000+. This explains why the data type of the column is object, when it should be numeric.\
To address this, I will:
- **Create Flag Column** - create a column named `flag_rough_estimate_hp` which flags the columns which are rough estimates rather than exact values.
- **Create Clean Column** - create a clean column which removes any non-numeric characters, and converts into a number column using `pd.to_numeric()`.
- **Assess Rough Estimates** - check rough estimates for a pattern e.g. same model or make.  

In [13]:
# unique values 
print("Below is all the unique horsepower values in the dataset:")
print(df.horsepower.unique())


# data type 
print("\nThe data type of the horsepower column is:")
print(df.horsepower.dtype)

Below is all the unique horsepower values in the dataset:
['379' '630' '661' '562' '710' '617' '523' '490' '760' '600' '1500' '717'
 '296' '1280' '471' '416' '454' '300' '505' '320' '626' '671' '622' '720'
 '1914' '414' '759' '986' '591' '503' '650' '660' '350' '641' '611' '394'
 '612' '369' '603' '455' '460' '325' '349' '592' '444' '405' '797' '770'
 '332' '473' '480' '573' '380' '1600' '181' '620' '764' '624' '1000+'
 '382' '800' '715' '690' '730' '469' '365' '401' '645' '435' '1020' '500'
 '780' '750' '402' '575' '729' '789' '577' '495' '237' '310' '791' '1874'
 '542' '368' '616' '1479' '755' '1,000+' '288' '1888' '10000+' '482'
 '1973' '1262' '1035' '819' '385' '647' '1200' '1578' '625' '583' '429'
 '563' '400' '707' '887' '1972' '305' '640' '255' '689' '372' '1000'
 '2000' '550' '10,000' '1,500' '10,000+' '485' '1,020' '1872' '621']

The data type of the horsepower column is:
object


I created the cleaned the horsepower column, by removing the plus sign and then using `pd.to_numeric()` to convert to an integer. 

In [14]:

# remove plus and commas - create horsepower_clean
df['horsepower_clean'] = df['horsepower'].astype(str).str.rstrip('+').str.replace(",", "")


# convert to numeric 
df["horsepower_clean"] = pd.to_numeric(df["horsepower_clean"], errors="coerce")

# dtype and missing 

print("The number of missing values in the clean column is now:")
print(df.horsepower_clean.isna().sum())
print("\nThe data type of the clean column is now:")
print(df.horsepower_clean.dtype)

# side by side of old and new column for hp
print(df[["horsepower", "horsepower_clean"]][90:100])

The number of missing values in the clean column is now:
0

The data type of the clean column is now:
int64
    horsepower  horsepower_clean
93         620               620
94         622               622
95         764               764
96         414               414
98         624               624
99       1000+              1000
100        382               382
101        800               800
102        300               300
103        770               770


Next I flagged and assessed all of the columns. The flagged column is a boolean column which contains True if it meets one of two conditions: original horsepower column contains a plus sign (+) or the horsepower value is unreasonable (i.e. greater than 2500). 
I then assessed all of the rows with a rough estimate (those with True - on the flag column), and discovered that they all appear to be Tesla Roadsters. I also noticed that many of them seem to have the same production year, horsepower and price - this can be looked into later on when removing duplicates (second pass). \
While looking at this data I noticed that there were some values with 10,000 horsepower, which is not reasonable (suggesting possible errors while inputting data). There were three cars which claimed to have 10,000 which were the Tesla Roadster 2022. A quick search discovered that the actual horsepower of these cars is around 1,000, which reinforces the idea that there was an issue in inputting data (added an extra zero by mistake). This will be recified, changing the two 10,000 hp into 1,000. I then doubled checked that the values had been replaced successfully. 
Now that there horsepower column has been cleaned, assessed for missing values (there are none) and the data type is correct, I can move onto the next column. 

In [15]:
# flag the rough estimates

df["flag_rough_estimate_hp"] = (
    df['horsepower'].astype(str).str.endswith('+') |
    (df['horsepower_clean'] >= 2500)
)


# assess all the rough estimates - appear to all be Tesla Roadsters
print(df[df["flag_rough_estimate_hp"] == True])


# notice odd outliers - 10,000 hp - unreasonably high and will be adjusted to 

print(df[df["horsepower_clean"] == 10000])

# replace 10000 

df["horsepower_clean"] = df["horsepower_clean"].replace(10000, 1000)

# check again  successfully removed 

print(df[df["make"] == "Roadster"])

print(df[df["horsepower_clean"] == 10000])



      make     model  year engine_size_l horsepower torque_lb zero_to_sixty  \
99   Tesla  Roadster  2022      Electric      1000+       737           1.9   
335  Tesla  Roadster  2022             -      1000+         -           1.9   
354  Tesla  Roadster  2022      Electric      1000+   10,000+           1.9   
364  Tesla  Roadster  2023      Electric     1,000+       737         < 1.9   
389  Tesla  Roadster  2022           NaN     10000+         0           1.9   
885  Tesla  Roadster  2022             0     10,000     7,376           1.9   
916  Tesla  Roadster  2022           NaN    10,000+       NaN           1.9   

    price_usd year_clean engine_size_clean  numeric_engine_size engine_type  \
99    200,000       2022          Electric                  NaN    Electric   
335   200,000       2022               NaN                  NaN    Electric   
354   200,000       2022          Electric                  NaN    Electric   
364   200,000       2023          Electric         

### Assess `torque_lb` Column 
The next column to assess is the `torque_lb` which is the rotational force generated by the engine (lb-feet). Viewing the unqiue values, data type and missing value number shows a few issues which must be rectified:
- **Placeholder Values** - there are some values with zero or a dash inplace of NaN, which must be removed to get an accurate value for missingness.
- **Characters** - there are some non-numeric characters within the values e.g. plus signs, commas.
- **Incorrect Datatype** - the data type is object, when it should be numeric. This is likely due to the placeholder values and non-numeric characters (+ and ,). 
- **Missing Values** - there are some values which are missing (and possibly more once placeholders are replaced), which must be assessed.

In [16]:
# unique values 
print("The unique torque values are:")
print(df.torque_lb.unique())

# missing values 
print("\nThe number of missing values is:")
print(df.torque_lb.isna().sum())

# data type 
print("\nThe data type is:")
print(df.torque_lb.dtype)

The unique torque values are:
['331' '443' '561' '406' '568' '553' '494' '465' '625' '481' '516' '1180'
 '656' '295' '1015' '398' '317' '384' '280' '243' '664' '531' '468' '737'
 '738' '1696' '309' '590' '479' '650' '550' '276' '626' '369' '420' '627'
 '455' '505' '560' '457' '707' '270' '354' '476' '339' '1106' '151' '605'
 '368' '723' '642' '509' '604' '507' '513' '600' '440' '1050' '708' '774'
 '254' '663' '332' '530' '470' '258' '290' '413' '1732' '376' '-'
 '10,000+' '236' '0' '472' '1254' '848' '1300' '442' '641' '498' '350' nan
 '944' '268' '184' '400' '263' '7,376' '1,180' '475' '1,050' '740' '538']

The number of missing values is:
3

The data type is:
object


### Remove placeholders, non-numeric characters and convert to numeric
I removed the placeholder values (0 and -) and then removed any non-numeric characters (+ or ,). I then converted this to a numeric variable (Int64 - because there are missing values). 

In [17]:


# remove placeholders 
placeholder = ["0", "-"]

df["torque_clean"] = df["torque_lb"].replace(placeholder, np.NaN)

# remove non-numeric characters 

df['torque_clean'] = df['torque_clean'].astype(str).str.rstrip('+').str.replace(",", "")

print(df.torque_clean.unique())

# convert to numeric 

df["torque_clean"] = pd.to_numeric(df["torque_clean"], errors="coerce").astype("Int64")


# side by side 

print(df[["torque_lb", "torque_clean"]].sample(5))

['331' '443' '561' '406' '568' '553' '494' '465' '625' '481' '516' '1180'
 '656' '295' '1015' '398' '317' '384' '280' '243' '664' '531' '468' '737'
 '738' '1696' '309' '590' '479' '650' '550' '276' '626' '369' '420' '627'
 '455' '505' '560' '457' '707' '270' '354' '476' '339' '1106' '151' '605'
 '368' '723' '642' '509' '604' '507' '513' '600' '440' '1050' '708' '774'
 '254' '663' '332' '530' '470' '258' '290' '413' '1732' '376' 'nan'
 '10000' '236' '472' '1254' '848' '1300' '442' '641' '498' '350' '944'
 '268' '184' '400' '263' '7376' '475' '740' '538']
    torque_lb  torque_clean
440       280           280
906       295           295
256       420           420
858       479           479
494       354           354


The next step was to create a rough estimate column which is a boolean column. This will flag true if the data meets any of these conditions: original torque column contains a plus (+) or the torque value is unreasonable (i.e. greater than 2500). After viewing these, I noticed that both were tesla roadsters, one claiming a torque of over 10,000 and the other claiming a torque of 7,373. A quick search discovered that the 7,376 is actually a value for wheel torque of the tesla roadster (which is generally much higher than actual motor torque). There is no actual motor torque values reported for the tesla roadster 2022, so I will just replace the values with NA. 

In [18]:
# flag rough estimates
df["flag_rough_estimate_torque"] = df['torque_lb'].astype(str).str.endswith('+') | (df["torque_clean"] >= 2500)

# view rough estimates 
print(df[df["flag_rough_estimate_torque"] == True])

# replace with NA

df['torque_clean'] = df['torque_clean'].replace({10000: np.nan, 7376: np.nan})


      make     model  year engine_size_l horsepower torque_lb zero_to_sixty  \
354  Tesla  Roadster  2022      Electric      1000+   10,000+           1.9   
885  Tesla  Roadster  2022             0     10,000     7,376           1.9   

    price_usd year_clean engine_size_clean  numeric_engine_size engine_type  \
354   200,000       2022          Electric                  NaN    Electric   
885   200,000       2022               NaN                  NaN    Electric   

     horsepower_clean  flag_rough_estimate_hp  torque_clean  \
354              1000                    True         10000   
885              1000                    True          7376   

     flag_rough_estimate_torque  
354                        True  
885                        True  


### Assessed missing values 
Next I looked at the missing values, finding that many of those with missing torque values are Tesla Roadsters, Tesla Model S Plaid and a Maserati GranTurismo. There are imputation methods which can be used, such as finding the mean or median value (by make/ model/ engine size/ engine type), or even by looking up the engine torque on the Tesla and Maserati website. A possible reason for missing values in Tesla vehicles may be because they tend to report wheel torque rather than motor torque, which might explain why there are many missing values. \
I am going to leave the values as NA for now, but make note that there are missing values in the torque column. 

In [19]:
print(df[df.torque_clean.isna()])

         make          model  year engine_size_l horsepower torque_lb  \
335     Tesla       Roadster  2022             -      1000+         -   
354     Tesla       Roadster  2022      Electric      1000+   10,000+   
389     Tesla       Roadster  2022           NaN     10000+         0   
642     Tesla  Model S Plaid  2021      Electric       1020       NaN   
878  Maserati    GranTurismo  2021      Electric        550       NaN   
885     Tesla       Roadster  2022             0     10,000     7,376   
916     Tesla       Roadster  2022           NaN    10,000+       NaN   

    zero_to_sixty price_usd year_clean engine_size_clean  numeric_engine_size  \
335           1.9   200,000       2022               NaN                  NaN   
354           1.9   200,000       2022          Electric                  NaN   
389           1.9   200,000       2022               NaN                  NaN   
642           1.9   139,990       2021          Electric                  NaN   
878       

### Assessing `zero_to_sixty` Column 
The next step is to assess the `zero_to_sixty` column, which is the time it takes the car to go from 0-60 in seconds. Looking at the unique values, most are floats or integers, but there is one string "< 1.9" which explains why the data is saved as an object. I will remove this non-numeric character and convert to numeric. There was no missing data in this column. 

In [20]:
# unique values 
print("The unique values in this column are:")
print(df.zero_to_sixty.unique())

# missing values 

print("\nThe number of missing values in this column are:")

print(df.zero_to_sixty.isna().sum())

# data type 
print("\nThe data type of this column is:")
print(df.zero_to_sixty.dtype)


The unique values in this column are:
['4' '2.8' '3' '3.2' '2.7' '3.1' '3.8' '3.5' '2.5' '2.4' '5.4' '4.4' '4.8'
 '4.7' '3.6' '4.1' '1.85' '4.5' '3.3' '3.9' '4.2' '3.4' '5.1' '4.3' '2.9'
 '5' '5.3' '4.9' '6.5' '3.7' '1.9' '1.98' '2.6' '4.6' '2.3' '< 1.9' '1.8'
 '2.1' '5.2' '2.2' '6.4' '2']

The number of missing values in this column are:
0

The data type of this column is:
object


### Remove non-numeric characters and convert to numeric
Used `lstrip()` to remove the leading <  and then converted to numeric data (float64). I then checked there were still no missing values (there weren't which means that data was converted correctly) and the data type which was correctly saved as float64. Now that the zero to sixty column has the correct data type, I can move onto the next column. 

In [21]:
# flag rough estimates
df["flag_rough_estimate_zero_to_sixty"] = df['zero_to_sixty'].astype(str).str.startswith('<')

# remove non-numeric character (<)
df['zero_to_sixty_clean'] = df['zero_to_sixty'].astype(str).str.lstrip('< ')

# convert to numeric 

df["zero_to_sixty_clean"] = pd.to_numeric(df['zero_to_sixty_clean'], errors="coerce")


# check data 

print(df[["zero_to_sixty", "zero_to_sixty_clean"]].sample(5))

print("\nThe number of missing values:")
print(df.zero_to_sixty_clean.isna().sum())

print("\nData type is:")
print(df.zero_to_sixty_clean.dtype)

    zero_to_sixty  zero_to_sixty_clean
557           3.1                  3.1
310           3.2                  3.2
743           2.6                  2.6
514           2.3                  2.3
187           2.9                  2.9

The number of missing values:
0

Data type is:
float64


### Assess `price_usd` Column 
The final column of the data set to assess is the `price_usd` column, which contains the price of each car in US dollars. An overview of the data shows that there are no missing values and the data type is incorrect (object when it should be a numeric). A view of the first 50 unique values shows that the incorrect data type is likely due to most being saved with a comma (presence of a non-numeric character leads to being saved as object). \
- **Placeholders** - I will quickly remove any possible placeholders like 0 or a dash (doesn't appear to be any but as there are so many unique values I will do this just incase).
-**Remove non-numeric characters** - will then remove any non-numeric characters like commas. which will allow the column to be converted into a numeric format. 
-   **Convert to Numeric** - I will use `pd.to_numeric` to change the data type of the column from object to numeric.
-   **Create New price_gbp Column** - I will then create a new column which contains the price of each car in GB pounds. This will be done using the current exchange rate (0.76 pound to 1 dollar) in a lambda function. Creating this column will mean that the data and results are more applicable to a british audience (can also include the US dollar amount to widen the audience - applies to both).

In [22]:
# unique values 
print("Number of unique values:")
print(df.price_usd.nunique())
print("\nThe unique values in the price column are:")
print(df.price_usd.unique()[:50])

# missing values 
print("\nThe number of missing values is:")
print(df.price_usd.isna().sum())

# data type
print("\nThe type of data is:")
print(df.price_usd.dtype)

Number of unique values:
367

The unique values in the price column are:
['101,200' '274,390' '333,750' '142,700' '298,000' '130,000' '118,500'
 '59,900' '81,000' '212,000' '201,495' '3,000,000' '61,000' '70,100'
 '2,800,000' '92,950' '104,450' '150,000' '62,000' '78,000' '75,000'
 '225,000' '248,000' '155,000' '2,400,000' '100,200' '517,770' '625,000'
 '117,000' '72,800' '222,000' '64,695' '500,000' '45,690' '104,000'
 '218,000' '57,000' '210,000' '148,500' '132,000' '58,900' '518,000'
 '56,200' '192,500' '71,800' '68,000' '42,500' '39,000' '46,100' '142,100']

The number of missing values is:
0

The type of data is:
object


### Remove placeholders and non-numeric characters, and convert to numeric column
Replaced any placeholders (zero or dash) and removed any non-numeric characters (+ or a ,). I then checked the missing values, and found there was none (which confirms that there were no placeholders in the data). I also looked at a side by side comparison of the column, which showed that non-numeric characters were removed correctly (mainly the commas). Next I converted the column into numeric (float64). This means that I can move onto the final step, which is creating the new `price_gbp` column. 

In [23]:
# remove placeholders 

placeholder = ["0", "-"]

df["price_usd_clean"] = df["price_usd"].replace(placeholder, np.NaN)

# remove non-numeric characters (+ or ,)

df['price_usd_clean'] = df['price_usd_clean'].astype(str).str.rstrip('+').str.replace(",", "")

# check data 
print("The number of missing values after placeholders and non-numeric removed:")
print(df.price_usd_clean.isna().sum())

print("\nSide by side comparison of old vs new")
print(df[["price_usd", "price_usd_clean"]].sample(5))

# convert to numeric 

df['price_usd_clean'] = pd.to_numeric(df['price_usd_clean'], errors="coerce").astype("float64")

print("\nData type after conversion:")
print(df.price_usd_clean.dtype)

The number of missing values after placeholders and non-numeric removed:
0

Side by side comparison of old vs new
     price_usd price_usd_clean
682    310,000          310000
804    162,000          162000
455     69,900           69900
541  5,200,000         5200000
243    150,400          150400

Data type after conversion:
float64


### Create New `price_gbp_clean` column 
Finally I will create a new price column for GB pounds. To do this I will create a lambda function which calculates the GBP amount, using the `price_usd_clean` with an exchange rate (in this case I am using 0.78 - however this is constantly changing so may not be accurate at time of reading). I will apply this lambda function to the dataframe, and have the results saved into a new column called `price_gdp_clean`. \
After this I checked that there were no missing values (which there weren't), and the data type which was correctly saved as float64. Finally I viewed the two columns side by side, which showed that the conversion appears to be correct and column is formatted properly. 

In [24]:
# create lambda function and apply to column (create new column)

conversion_lambda = lambda row: row["price_usd_clean"] * 0.78

df["price_gbp_clean"] = df.apply(conversion_lambda, axis=1)

# view old and new column 
print(df[["price_usd_clean", "price_gbp_clean"]].sample(5))

# missing values 
print("\nThe missing values in new column:")
print(df.price_gbp_clean.isna().sum())

# data type 
print("\nThe data type of the new column:")
print(df.price_gbp_clean.dtype)

     price_usd_clean  price_gbp_clean
521          63100.0          49218.0
718         417650.0         325767.0
48          192500.0         150150.0
642         139990.0         109192.2
322          82190.0          64108.2

The missing values in new column:
0

The data type of the new column:
float64


In [25]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 641 entries, 0 to 1006
Data columns (total 20 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   make                               641 non-null    object  
 1   model                              641 non-null    object  
 2   year                               641 non-null    int64   
 3   engine_size_l                      632 non-null    object  
 4   horsepower                         641 non-null    object  
 5   torque_lb                          638 non-null    object  
 6   zero_to_sixty                      641 non-null    object  
 7   price_usd                          641 non-null    object  
 8   year_clean                         641 non-null    category
 9   engine_size_clean                  630 non-null    object  
 10  numeric_engine_size                589 non-null    float64 
 11  engine_type                        641 non-null  

In [26]:
# identify duplicates 
duplicates_second = df.duplicated(subset=['make', 'model', 'year_clean', 'engine_size_clean', 'engine_type', 'price_usd_clean', 'horsepower_clean', 'torque_clean'])

print(duplicates_second.value_counts())

df[duplicates_second]

# drop dupliactes 
df.drop_duplicates(
    subset=['make', 'model', 'year_clean', 'engine_size_clean', 'engine_type', 'price_usd_clean', 'horsepower_clean', 'torque_clean'],
    keep='first',   # or 'last'
    inplace=True)

# view table after duplicates removed
print(df.info())

False    636
True       5
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 636 entries, 0 to 1006
Data columns (total 20 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   make                               636 non-null    object  
 1   model                              636 non-null    object  
 2   year                               636 non-null    int64   
 3   engine_size_l                      629 non-null    object  
 4   horsepower                         636 non-null    object  
 5   torque_lb                          634 non-null    object  
 6   zero_to_sixty                      636 non-null    object  
 7   price_usd                          636 non-null    object  
 8   year_clean                         636 non-null    category
 9   engine_size_clean                  628 non-null    object  
 10  numeric_engine_size                588 non-null    float64 
 1

## Create a Clean DataFrame
The next step is to create a clean DataFrame (`clean_df`) which includes only the columns which have been checked. I have included all of the cleaned columns in this, and the flagged estimates columns for torque, horsepower and 0-60. I haven't done anything with the four missing values in the torque column, but have made note of this for later analysis/ visualisation. I also haven't done anything with the missing values in the numeric engine size column, this is because the missing values are not random (MNAR), because they are all electric cars and therefore don't have an engine size (not in Cylinder Capacity Measurements). The missing values are structural, not accidental/ random missingness. 

In [27]:
clean_df = pd.DataFrame({"make": df["make"],
                        "model": df["model"],
                        "year": df["year_clean"],
                         "numeric_engine_size": df["numeric_engine_size"],
                         "engine_type": df["engine_type"],
                         "horsepower": df["horsepower_clean"],
                         "flag_hp_estimate": df["flag_rough_estimate_hp"],
                         "torque_lb": df["torque_clean"],
                         "flag_torque_estimate": df["flag_rough_estimate_torque"],
                         "zero_to_sixty": df["zero_to_sixty_clean"],
                         "flag_zero_to_sixty_estimate": df["flag_rough_estimate_zero_to_sixty"],
                         "price_usd": df["price_usd_clean"],
                         "price_gbp": df["price_gbp_clean"]
                        })

print(clean_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 636 entries, 0 to 1006
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   make                         636 non-null    object  
 1   model                        636 non-null    object  
 2   year                         636 non-null    category
 3   numeric_engine_size          588 non-null    float64 
 4   engine_type                  636 non-null    category
 5   horsepower                   636 non-null    int64   
 6   flag_hp_estimate             636 non-null    bool    
 7   torque_lb                    632 non-null    Int64   
 8   flag_torque_estimate         633 non-null    boolean 
 9   zero_to_sixty                636 non-null    float64 
 10  flag_zero_to_sixty_estimate  636 non-null    bool    
 11  price_usd                    636 non-null    float64 
 12  price_gbp                    636 non-null    float64 
dtypes: Int64(

In [28]:
clean_df.to_csv("sports_car_clean.csv", index=False)